In [ ]:
import random
import torch
import numpy as np
from models.convnext import load_model
from utils.plot_utils import interactive_plot_with_pred
from utils.dataset import  get_custom_dataset_path



In [ ]:
""" path """
# Base path of the validation dataset
BASE_PATH = '/mnt/shared-storage/shared/Glaciomega/Glaciomega2_valdataset'

# if you want to save prediction change None to a path
save_path_test = None


"""model"""
# multi or mono temporal modele
model_type = 'multi' 

# backbone weights ConvNeXt
pretrain_ConvNeXtDPT= 'weights/convNext_base_DPT_finetune.pth' 

# Temporal weight of DPTHeadTemporal
# if you want to use MTPE (best results over Glacioclim Dataset)
pretrain_DPTHeadTemporal = 'weights/convNext_base_DPT_multitemp_MultiRelativePos.pth' 
encoder_type = 'gla3'
# if you want to us TPE (best results over validation dataset)
pretrain_DPTHeadTemporal = 'weights/convNext_base_DPT_multitemp_SimpleRelativePos.pth'
encoder_type = 'gla2'

""" seed """
seed = 42

""" device """
device = 'cuda'


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
seed_everything(seed)

In [ ]:
# load model
model_ConvNeXtDPT, temporal_head = load_model( model_type, encoder_type, pretrain_ConvNeXtDPT, pretrain_DPTHeadTemporal, use_weight_from_DPT=False, freeze_bbone=True, device=device )


In [ ]:
# load the dataset
data_names = ['konarak',
            'glacier_cham',
            'criel-sur-mer',
            'Tonga-Hunga_Ha_apai',
            'ossoue',
            'soulac',
            'Natanz',
            'Marinka',
            'newtok',
            'cadelaria',
            'bondo',
            'bessat',
            'gaza',
            'piton_fournaise',
            'zone_combat_1',
            'kilaueae',
            'Bakhmut',
            'Vuhledar',
            'blatten',
            'Alex',
            'mariupol']

dataset = get_custom_dataset_path( BASE_PATH, data_names )

In [ ]:
infer_size = 512 # path size for inference
step_size = 128 # overlap size of patch

# use_time_forward controll how we pass on image
# False: image are seen once : 0 to 8, 8 to 16, 16 to 24...
# True: image are seen multiple time with a strategy enabling the model to have a larger temporal window
# For exemple with a batch of 8, following indexes will be seen :
# [0 1 2 3 4 5 6 7]
# [ 0  2  4  6  8  9 10 11]
# [ 0  3  6  9 12 13 14 15]
# [ 0  4  8 12 16 17 18 19]
# [ 0  5 10 15 20 21 22 23]
use_time_forward = False
max_bs = 8 # batch size



In [ ]:
%matplotlib widget

# interactive plot, you can use the tools from the widget to zoom in, change the dataset to show etc..
interactive_plot_with_pred(dataset, 
                model_ConvNeXtDPT=model_ConvNeXtDPT,
                temporal_head=temporal_head,
                infer_size=infer_size,
                step_size=step_size,
                use_time_forward=use_time_forward,
                max_bs=max_bs,
                p_out=save_path_test,
                encoder_type=encoder_type,
                device=device,
                model_type=model_type,
            )
